# Surrogate Model Analysis

Produces the paper's surrogate-model results (`§Can classical opinion-dynamics
models explain these dynamics?`): final-state MCC and consensus fidelity for
the M1-M4 surrogate hierarchy (Fig. "pooled-mcc", Fig. "pooled-consensus",
Table "final-mcc", Table "consensus-fidelity").

**This notebook only pools and visualizes results**:
1.  the surrogate models themselves (M1: persistence, M2: persistence + global belief composition,
 M3: persistence + local neighborhood composition, M4: M3 + agent identity; see
`Appendix: Surrogate Model Details`) are fit, evaluated (held-out MCC), and
rolled out (consensus fidelity) by a separate pipeline in
`src/analysis/dynamics_model_fitting/`, which writes its outputs (per-model
`*__results.json` and `*__trajectory_behavior_summary.parquet` files) to
`src/analysis/dynamics_model_fitting/outputs/`. This notebook discovers those
files, aggregates across runs, and builds the manuscript-ready figures/tables.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl


## Locate the surrogate-fitting outputs

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src/analysis/dynamics_model_fitting/outputs").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing dynamics_model_fitting outputs.")


PROJECT_ROOT = find_project_root(Path.cwd())
DMF_DIR = PROJECT_ROOT / "src/analysis/dynamics_model_fitting"
OUTPUTS_DIR = DMF_DIR / "outputs"
FIG_DIR = OUTPUTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUTS_DIR:", OUTPUTS_DIR)


## 1. Consensus fidelity

Each surrogate is initialized from the empirical round-0 belief state and
rolled out stochastically; consensus fidelity is `1 - mean absolute error`
between the surrogate's and the empirical simulation's final-round consensus
(Eq. "consensus fidelity"). We pool this over the held-out test runs, 
per (surrogate, scenario, network).


In [ ]:
BEHAV_RE = re.compile(
    r"(?P<model>m[\d-]+)__exp-(?P<experiment_group>[^_]+(?:_[^_]+)*)__graph-(?P<graph_type>[^_]+(?:-[^_]+)*)__"
    r"trans-(?P<exclude>false|true)__split-(?P<split>train|val|test)__match-(?P<match_mode>latest|all)__trajectory_behavior_summary\.parquet$"
)


def collect_behavior_metrics(fit_mode: str) -> pd.DataFrame:
    """Load every trajectory-behavior summary file for a given fit mode into one long table."""
    rows: list[dict] = []
    fit_root = OUTPUTS_DIR / fit_mode

    for path in sorted(fit_root.glob("m*/trajectory_behavior/*__trajectory_behavior_summary.parquet")):
        m = BEHAV_RE.search(path.name)
        if not m:
            continue

        summary = pd.read_parquet(path)
        if summary.empty or "rollout_mode" not in summary.columns:
            continue

        base = {
            "fit_mode": fit_mode,
            "model": m.group("model"),
            "experiment_group": m.group("experiment_group"),
            "graph_type": m.group("graph_type"),
            "exclude_round0_to_1": m.group("exclude") == "true",
            "split": m.group("split"),
            "match_mode": m.group("match_mode"),
            "summary_path": str(path),
            "file_mtime": path.stat().st_mtime,
        }

        metric_cols = [c for c in summary.columns if c != "rollout_mode" and pd.api.types.is_numeric_dtype(summary[c])]
        for _, r in summary.iterrows():
            for metric in metric_cols:
                value = r[metric]
                if pd.isna(value):
                    continue
                rows.append({**base, "rollout_mode": r["rollout_mode"], "metric": metric, "value": float(value)})

    return pd.DataFrame(rows)


behavior_df_raw = pd.concat(
    [collect_behavior_metrics("single_transitions"), collect_behavior_metrics("full_trajectories")],
    ignore_index=True,
)

# Keep only the four models reported in the paper (M1-M4);
behavior_key_cols = [
    "fit_mode", "model", "experiment_group", "graph_type",
    "exclude_round0_to_1", "split", "rollout_mode", "metric",
]
surrogate_to_keep = ["m1", "m2-5", "m3", "m4"]

behavior_df = (
    behavior_df_raw.sort_values("file_mtime")
    .drop_duplicates(subset=behavior_key_cols, keep="last")
    .loc[lambda d: d["model"].isin(surrogate_to_keep)]
    .reset_index(drop=True)
)

print("Behavior rows (raw / deduplicated to M1-M4):", len(behavior_df_raw), len(behavior_df))


In [ ]:
# Consensus fidelity slice used for the paper's figures: full-trajectory fits,
# held-out test runs, stochastic rollout, and all transitions (no round-0-to-1 exclusion).
consensus_metrics = [
    "all_runs__consensus_fraction_abs_error__point",
    "all_runs__consensus_fraction_abs_error__ci_lower",
    "all_runs__consensus_fraction_abs_error__ci_upper",
]

consensus_df_raw = (
    behavior_df_raw.loc[
        (behavior_df_raw["split"] == "test")
        & (~behavior_df_raw["exclude_round0_to_1"])
        & (behavior_df_raw["rollout_mode"] == "stochastic")
        & (behavior_df_raw["match_mode"] == "latest")
        & (behavior_df_raw["fit_mode"] == "full_trajectories")
        & (behavior_df_raw["metric"].isin(consensus_metrics)),
        ["model", "experiment_group", "graph_type", "summary_path", "metric", "value"],
    ]
    .pivot(index=["model", "experiment_group", "graph_type", "summary_path"], columns="metric", values="value")
    .reset_index()
    .dropna(subset=consensus_metrics)
)

consensus_df_agg = (
    consensus_df_raw.groupby(["model", "experiment_group", "graph_type"], as_index=False)
    .agg(
        **{
            "all_runs__consensus_fraction_abs_error__point": ("all_runs__consensus_fraction_abs_error__point", "mean"),
            "all_runs__consensus_fraction_abs_error__ci_lower": (
                "all_runs__consensus_fraction_abs_error__ci_lower",
                lambda s: float(np.percentile(s.to_numpy(dtype=float), 2.5)),
            ),
            "all_runs__consensus_fraction_abs_error__ci_upper": (
                "all_runs__consensus_fraction_abs_error__ci_upper",
                lambda s: float(np.percentile(s.to_numpy(dtype=float), 97.5)),
            ),
            "n_runs": ("summary_path", "nunique"),
        }
    )
    .sort_values(["model", "experiment_group", "graph_type"])
)

# Consensus fidelity = 1 - (absolute error between surrogate and empirical consensus).
consensus_lfs = pl.from_pandas(consensus_df_agg).with_columns(
    (1 - pl.col("all_runs__consensus_fraction_abs_error__point")).alias("consensus_agreement_point"),
    (1 - pl.col("all_runs__consensus_fraction_abs_error__ci_upper")).alias("consensus_agreement_ci_lower"),
    (1 - pl.col("all_runs__consensus_fraction_abs_error__ci_lower")).alias("consensus_agreement_ci_upper"),
)
consensus_lfs


## 2. Final-state MCC

Held-out one-step predictive performance (Matthews Correlation Coefficient)
for the same slice (full-trajectory fits, held-out test runs, all
transitions).


In [ ]:
FIT_RE = re.compile(
    r"(?P<model>m[\d-]+)__exp-(?P<experiment_group>[^_]+(?:_[^_]+)*)__graph-(?P<graph_type>[^_]+(?:-[^_]+)*)__trans-(?P<transition_tag>alltrans|exclude01)__"
    r"(?P<timestamp>\d{8}_\d{6})__results\.json$"
)


def _as_point_ci(metric_payload: object) -> tuple[float | None, float, float]:
    if isinstance(metric_payload, dict):
        point = metric_payload.get("point")
        if point is None:
            return None, np.nan, np.nan
        ci_lower, ci_upper = metric_payload.get("ci_lower"), metric_payload.get("ci_upper")
        return float(point), np.nan if ci_lower is None else float(ci_lower), np.nan if ci_upper is None else float(ci_upper)
    if isinstance(metric_payload, (int, float)):
        return float(metric_payload), np.nan, np.nan
    return None, np.nan, np.nan


def _extract_mcc_with_ci(payload: dict) -> tuple[float | None, float, float]:
    """Find `final_state_mcc` wherever it lives in a results.json payload."""
    test_payload = payload.get("metrics", {}).get("test")
    if not isinstance(test_payload, dict):
        return None, np.nan, np.nan

    traj = test_payload.get("trajectory_fit_metrics")
    if isinstance(traj, dict) and "final_state_mcc" in traj:
        point, lo, hi = _as_point_ci(traj["final_state_mcc"])
        if point is not None:
            return point, lo, hi

    trans = test_payload.get("transition_metrics")
    if isinstance(trans, dict):
        for k in ["final_state_mcc", "mcc"]:
            if k in trans:
                point, lo, hi = _as_point_ci(trans[k])
                if point is not None:
                    return point, lo, hi

    # Last-resort recursive search.
    stack = [test_payload]
    while stack:
        node = stack.pop()
        if isinstance(node, dict):
            for k, v in node.items():
                if k in {"final_state_mcc", "mcc"}:
                    point, lo, hi = _as_point_ci(v)
                    if point is not None:
                        return point, lo, hi
                if isinstance(v, dict):
                    stack.append(v)

    return None, np.nan, np.nan


mcc_rows = []
for path in sorted((OUTPUTS_DIR / "full_trajectories").glob("m*/**/*__results.json")):
    m = FIT_RE.search(path.name)
    if not m or m.group("transition_tag") != "alltrans":
        continue

    with path.open("r", encoding="utf-8") as f:
        payload = json.load(f)

    mcc_point, mcc_ci_lower, mcc_ci_upper = _extract_mcc_with_ci(payload)
    if mcc_point is None:
        continue

    mcc_rows.append({
        "model": m.group("model"), "experiment_group": m.group("experiment_group"),
        "graph_type": m.group("graph_type"), "timestamp": m.group("timestamp"),
        "mcc_point": mcc_point, "mcc_ci_lower": mcc_ci_lower, "mcc_ci_upper": mcc_ci_upper,
    })

mcc_df = pd.DataFrame(mcc_rows)
if not mcc_df.empty:
    mcc_df = (
        mcc_df.groupby(["model", "experiment_group", "graph_type"], as_index=False)
        .agg(
            mcc_point=("mcc_point", "mean"),
            mcc_ci_lower=("mcc_ci_lower", lambda s: float(np.percentile(s.to_numpy(dtype=float), 2.5))),
            mcc_ci_upper=("mcc_ci_upper", lambda s: float(np.percentile(s.to_numpy(dtype=float), 97.5))),
            n_runs=("mcc_point", "size"),
        )
        .sort_values(["model", "experiment_group", "graph_type"])
        .reset_index(drop=True)
    )

mcc_df.head()


## 3. Figures

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

from plot_utils import _configure_fonts, OKABE_ITO, GRAPH_COLOR, GRAPH_MAP

_configure_fonts()

SETTING_MAP = {
    "base_llms": "I. Base LLMs",
    "random_roles": "II. Random roles",
    "random_experts": "III. Spcs (random)",
    "experts": "IV. Spcs (matched)",
}
OPMOD_MAP = {"m1": "M1", "m2-5": "M2", "m3": "M3", "m4": "M4"}

plot_df_graph = (
    consensus_lfs.select(["model", "experiment_group", "graph_type", "consensus_agreement_point",
                           "consensus_agreement_ci_lower", "consensus_agreement_ci_upper"])
    .to_pandas()
    .merge(
        mcc_df[["model", "experiment_group", "graph_type", "mcc_point", "mcc_ci_lower", "mcc_ci_upper"]],
        on=["model", "experiment_group", "graph_type"], how="left",
    )
)
plot_df_graph.to_csv(FIG_DIR / "pooled_plot_data_merged.csv", index=False)

setting_order = ["base_llms", "random_roles", "random_experts", "experts"]
graph_order = ["erdos-renyi", "watts-strogatz"]
model_order = [m for m in ["m1", "m2-5", "m3", "m4"] if m in plot_df_graph["model"].unique()]
x = np.arange(len(model_order))
offsets = {"erdos-renyi": -0.07, "watts-strogatz": 0.07}

mcc_min = float(plot_df_graph["mcc_point"].min()) if plot_df_graph["mcc_point"].notna().any() else -1.0
mcc_max = float(plot_df_graph["mcc_point"].max()) if plot_df_graph["mcc_point"].notna().any() else 1.0
mcc_pad = max(0.02, 0.08 * (mcc_max - mcc_min if mcc_max > mcc_min else 0.2))
mcc_ylim = (max(-1.0, mcc_min - mcc_pad), 1.0)

graph_handles = [
    Line2D([0], [0], color=GRAPH_COLOR[g], marker="o", linestyle="-", lw=2.2, ms=7, label=GRAPH_MAP[g])
    for g in graph_order
]


def _style_axis(ax):
    ax.set_xticks(x)
    ax.set_xticklabels([OPMOD_MAP.get(m, m) for m in model_order])
    ax.grid(axis="y", alpha=0.25, linestyle="--", linewidth=1)
    ax.grid(axis="x", visible=False)
    for xv in x:
        ax.axvline(x=xv, color=OKABE_ITO["black"], linestyle=(0, (1.0, 3.0)), linewidth=1.0, alpha=0.40, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(OKABE_ITO["black"])
    ax.spines["bottom"].set_color(OKABE_ITO["black"])
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)
    ax.tick_params(axis="both", which="major", direction="out", width=1.3, length=6,
                    color=OKABE_ITO["black"], labelcolor=OKABE_ITO["black"])


def plot_2x2(df, y_col, yerr_lo_col, yerr_hi_col, ylabel, ylim, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(6.5, 6.5))
    panel_labels = ["A", "B", "C", "D"]

    for i, setting in enumerate(setting_order):
        ax = axes[i // 2, i % 2]
        panel = df[df["experiment_group"] == setting]
        for graph in graph_order:
            sub = panel[panel["graph_type"] == graph].set_index("model").reindex(model_order).reset_index()
            xs = x + offsets[graph]
            y = sub[y_col].to_numpy(dtype=float)
            lo, hi = y - sub[yerr_lo_col].to_numpy(dtype=float), sub[yerr_hi_col].to_numpy(dtype=float) - y
            if np.isfinite(lo).any() and np.isfinite(hi).any():
                ax.errorbar(xs, y, yerr=[lo, hi], fmt="o-", color=GRAPH_COLOR[graph], lw=2.0, ms=4, capsize=4, alpha=0.95)
            else:
                ax.plot(xs, y, "o-", color=GRAPH_COLOR[graph], lw=2.0, ms=4, alpha=0.95)

        ax.set_title(SETTING_MAP.get(setting, setting))
        _style_axis(ax)
        px = -0.24 if i % 2 else -0.30
        ax.text(px, 1.10, panel_labels[i], transform=ax.transAxes, ha="left", va="bottom",
                fontsize=14, fontweight="bold", color=OKABE_ITO["black"])
        ax.set_ylim(ylim)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))

    for row in range(2):
        axes[row, 0].set_ylabel(ylabel)

    fig.legend(graph_handles, [h.get_label() for h in graph_handles], loc="lower center",
               bbox_to_anchor=(0.5, -0.03), ncol=2, frameon=False)
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.14, wspace=0.3)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

In [ ]:
plot_2x2(
    plot_df_graph, y_col="consensus_agreement_point",
    yerr_lo_col="consensus_agreement_ci_lower", yerr_hi_col="consensus_agreement_ci_upper",
    ylabel="Consensus Fidelity", ylim=(0.68, 0.98), save_path=FIG_DIR / "consensus-fidelity.pdf",
)

plot_2x2(
    plot_df_graph, y_col="mcc_point",
    yerr_lo_col="mcc_ci_lower", yerr_hi_col="mcc_ci_upper",
    ylabel="Final-state MCC", ylim=mcc_ylim, save_path=FIG_DIR / "final-state-mcc.pdf",
)

## 4. LaTeX tables (Table "final-mcc", Table "consensus-fidelity")

Both tables exclude the literal `"m2"` model tag: `m2-5` is the actual M2
surrogate **reported** in the paper, while a handful of runs were fit under a
now-superseded `m2` identifier from an earlier iteration of the pipeline and
must not be mixed into the M2 estimates or their CIs.


In [ ]:
save_path = Path("../data/analysis/surrogate/")
save_path.mkdir(parents=True, exist_ok=True)

setting_short = {
    "base_llms": "I. Baseline", "random_roles": "II. Random Roles",
    "random_experts": "III. Random specialists", "experts": "IV. Specialists (matched)",
}
model_short = {"m1": "M1", "m2-5": "M2", "m3": "M3", "m4": "M4"}
network_short = {"erdos-renyi": "ER", "watts-strogatz": "WS"}


def build_surrogate_table(
    df: pd.DataFrame, metric_point: str, metric_ci_low: str, metric_ci_high: str, metric_label: str,
    include_network: bool = True, percent_scale: bool = True, digits: int = 1,
    caption: str = "", label: str = "",
) -> tuple[pd.DataFrame, str]:
    work = df.copy()
    work["Surrogate"] = work["model"].map(model_short).fillna(work["model"])
    work["Scenario"] = work["experiment_group"].map(setting_short).fillna(work["experiment_group"])
    work["Network"] = work["graph_type"].map(network_short).fillna(work["graph_type"])

    scale = 100.0 if percent_scale else 1.0
    work[[metric_point, metric_ci_low, metric_ci_high]] *= scale

    group_cols = ["Surrogate", "Scenario"] + (["Network"] if include_network else [])
    out = (
        work.groupby(group_cols, as_index=False)
        .agg({metric_point: "mean", metric_ci_low: "mean", metric_ci_high: "mean"})
        .rename(columns={metric_point: metric_label, metric_ci_low: "CI low", metric_ci_high: "CI high"})
    )

    surrogate_order = ["M1", "M2", "M3", "M4"]
    scenario_order = list(setting_short.values())
    out["Surrogate"] = pd.Categorical(out["Surrogate"], categories=surrogate_order, ordered=True)
    out["Scenario"] = pd.Categorical(out["Scenario"], categories=scenario_order, ordered=True)
    if include_network:
        out["Network"] = pd.Categorical(out["Network"], categories=["ER", "WS"], ordered=True)
    out = out[out["Surrogate"].notna() & out["Scenario"].notna()].copy()

    sort_cols = ["Scenario", "Surrogate"] + (["Network"] if include_network else [])
    out = out.sort_values(sort_cols).reset_index(drop=True)
    for c in [metric_label, "CI low", "CI high"]:
        out[c] = out[c].map(lambda x: f"{x:.{digits}f}" if pd.notna(x) else "")

    latex = out.to_latex(index=False, escape=False, caption=caption, label=label)
    return out, latex


# Exclude the superseded "m2" tag before any aggregation (see note above).
consensus_src = consensus_lfs.to_pandas()
consensus_src = consensus_src.loc[consensus_src["model"] != "m2"].copy()

mcc_src = mcc_df.loc[mcc_df["model"] != "m2"].copy()

consensus_table, _ = build_surrogate_table(
    df=consensus_src, metric_point="consensus_agreement_point",
    metric_ci_low="consensus_agreement_ci_lower", metric_ci_high="consensus_agreement_ci_upper",
    metric_label="Consensus fidelity (%)", caption="Consensus fidelity by surrogate, scenario, and network.",
    label="tab:surrogate_consensus_fidelity",
)

mcc_table, _ = build_surrogate_table(
    df=mcc_src, metric_point="mcc_point", metric_ci_low="mcc_ci_lower", metric_ci_high="mcc_ci_upper",
    metric_label="Final-state MCC (%)", caption="Final-state MCC by surrogate, scenario, and network.",
    label="tab:surrogate_final_state_mcc",
)

mcc_table = mcc_table.sort_values(["Scenario", "Surrogate", "Network"]).set_index(["Scenario", "Surrogate", "Network"])
mcc_table.to_csv(save_path / "final_state_mcc.csv", index=True)

mcc_latex = mcc_table.to_latex(
    index=True, caption="", multicolumn=True, multirow=True, escape=False,
    float_format="{:0.1f}".format, label="tab:final-mcc",
)
mcc_latex = mcc_latex.replace("\\multirow[t]{", "\\multirow[c]{")
print(mcc_latex)

In [ ]:
consensus_table_indexed = (
    consensus_table.reset_index(drop=True)
    .sort_values(["Scenario", "Surrogate", "Network"])
    .set_index(["Scenario", "Surrogate", "Network"])
)
consensus_table_indexed.to_csv(FIG_DIR / "consensus_fidelity.csv", index=True)

consensus_latex = consensus_table_indexed.to_latex(
    index=True, caption="", multicolumn=True, multirow=True, escape=False, label="tab:consensus-fidelity",
)
consensus_latex = consensus_latex.replace("\\multirow[t]{", "\\multirow[c]{")
print(consensus_latex)
